# 04a — Principal Component Analysis (PCA)

**Employee Resignation Prediction Project**  
**Pipeline Stage:** Exploratory Dimensionality Reduction & Linear Benchmark Scaffold  
**Status:** SKELETON / READY FOR MANUAL IMPLEMENTATION

> [!WARNING]
> **CRITICAL LEAKAGE GUARD — DEVELOPMENT DATA ONLY**
>
> This notebook must use development data only (`partition == 'development'`, 80,000 records).
>
> The formal 20% holdout (20,000 records) must NOT be used for PCA fitting, scaling decisions, variance analysis, Logistic Regression comparison, feature decisions, or any other development-stage decision.

## 1. Purpose

Principal Component Analysis (PCA) is an unsupervised linear dimensionality-reduction technique that transforms a set of potentially correlated continuous and ordinal variables into a smaller set of orthogonal (uncorrelated) variables called **Principal Components (PCs)**.

### Key Technical Goals:
- Determine whether the 13 approved numerical/ordinal predictors can be effectively represented in a lower-dimensional subspace while retaining most of their variance.
- Unsupervised projection: PC1 captures the largest possible variance; subsequent components capture progressively less variance.
- Provide supporting linear benchmark evidence comparing Logistic Regression with and without PCA under 5-fold cross-validation.

## 2. PCA Methodology and Leakage Rules

### Predictor Scope & Input Contract:
**Approved 13 Numerical / Ordinal Predictors for PCA:**
1. `Age`
2. `Years_At_Company`
3. `Performance_Score`
4. `Monthly_Salary`
5. `Work_Hours_Per_Week`
6. `Projects_Handled`
7. `Overtime_Hours`
8. `Sick_Days`
9. `Remote_Work_Frequency`
10. `Team_Size`
11. `Training_Hours`
12. `Promotions`
13. `Employee_Satisfaction_Score`

**Strictly Excluded from Primary PCA Matrix:**
- `Employee_ID` (identifier)
- `Hire_Date` (redundant metadata)
- `Resigned` (target variable — PCA is strictly unsupervised; target is only used post-PCA for color coding or supervised evaluation)
- `Department`, `Gender`, `Job_Title`, `Education_Level` (the 4 categorical predictors are excluded from the primary continuous PCA)

**Leakage Constraints:**
- Only the 80,000 development rows may be loaded and analyzed.
- In supervised benchmarking (Section 20–23), `StandardScaler` and `PCA` must be fitted strictly fold-locally on the 4 training folds (64,000 rows) and applied to the validation fold (16,000 rows).

## 3. Imports and Setup

*Instruction:* Import required standard libraries (`pathlib`, `json`, `hashlib`, `yaml`, `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`, `sklearn.decomposition.PCA`, `sklearn.preprocessing.StandardScaler`, `sklearn.linear_model.LogisticRegression`, `sklearn.metrics`). Configure plotting styles, pandas formatting, and directory paths.

## 4. Load Dataset and Split Registry

*Instruction:* Load `data/raw/Extended_Employee_Performance_and_Productivity_Data.csv` and `data/processed/split_registry.csv`. Assert file checksums match verified hashes (`2b1e07...` for raw data, `87edc1...` for split registry).

## 5. Load Feature Contract

*Instruction:* Load `configs/feature_contract.yaml` and extract the approved list of numerical/ordinal feature names.

## 6. Select Development Partition

*Instruction:* Filter the split registry to `partition == 'development'` (80,000 records) and merge with raw data. Verify no holdout IDs are present.

## 7. Select the 13 Numerical / Ordinal Predictors

*Instruction:* Extract the feature matrix `X_dev` containing exactly the 13 numerical/ordinal columns, and separate the target vector `y_dev` (`Resigned`).

## 8. Validate PCA Inputs

*Instruction:* Assert that `X_dev` has shape `(80000, 13)`, contains zero null/missing values, and does not contain `Employee_ID`, `Hire_Date`, `Resigned`, or any categorical column.

## 9. Descriptive Statistics

*Instruction:* Generate a compact summary table of unscaled development features showing `count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`.

## 10. Standardize Features

*Instruction:* Standardize the 13 PCA predictors using `StandardScaler` fitted on the 80,000 development rows. Verify scaled mean ≈ 0 and std ≈ 1.

## 11. Fit PCA

*Instruction:* Fit `PCA(n_components=13, random_state=42)` on the standardized development data, retaining all 13 components.

## 12. Explained Variance

*Instruction:* Calculate individual explained variance ratio and eigenvalues for all 13 components. Create a clean table.

## 13. Cumulative Explained Variance

*Instruction:* Calculate cumulative explained variance ratio across components PC1 to PC13.

## 14. Components Required for 80%, 90%, and 95% Variance

*Instruction:* Determine programmatically the minimum number of principal components required to reach each cumulative explained-variance threshold (80%, 90%, 95%).

## 15. Scree Plot

*Instruction:* Generate a professional scree plot showing Component vs Individual Explained Variance (%). Save figure to `results/pca/scree_plot.png`.

## 16. Cumulative Variance Plot

*Instruction:* Generate a plot of Component Count vs Cumulative Explained Variance (%) with horizontal reference lines for 80%, 90%, and 95% thresholds. Save figure to `results/pca/cumulative_variance.png`.

## 17. Principal Component Loadings

*Instruction:* Compute the component loading matrix (features × components) from `pca.components_.T`.

## 18. Interpret PC1, PC2, and PC3

*Instruction:* Inspect the largest absolute loadings and signed values for PC1, PC2, and PC3. Provide an objective discussion of whether specific domain traits or broad mixtures emerge.

## 19. PC1 vs PC2 Visualization

*Instruction:* Create a 2D projection scatter plot of PC1 vs PC2, coloring points by `Resigned` (using a deterministic sample if needed for readability). Assess linear separability. Save figure to `results/pca/pc1_pc2_scatter.png`.

## 20. Logistic Regression Without PCA

*Instruction:* Define Pipeline A (`StandardScaler -> LogisticRegression(max_iter=1000, random_state=42)`). Prepare for 5-fold cross-validation.

## 21. Logistic Regression With PCA >=95%

*Instruction:* Define Pipeline B (`StandardScaler -> PCA(n_components=0.95, random_state=42) -> LogisticRegression(max_iter=1000, random_state=42)`).

## 22. Five-Fold Cross-Validation Comparison

*Instruction:* Execute 5-fold cross-validation for both Pipeline A and Pipeline B using the frozen fold assignments from `split_registry.csv`. Enforce strict fold-local fitting of scaler and PCA on training folds only. Generate 80,000 out-of-fold probability predictions.

## 23. Compare PR-AUC, ROC-AUC, and Brier Score

*Instruction:* Calculate pooled and fold-level metrics (PR-AUC, ROC-AUC, Brier score, Balanced Accuracy, Precision, Recall, F1 at base rate threshold 0.10010, runtime) for both pipelines. Present a comparison table.

## 24. Save PCA Results

*Instruction:* Save PCA tables and summary to `results/pca/`:
- `results/pca/explained_variance.csv`
- `results/pca/cumulative_variance.csv`
- `results/pca/component_loadings.csv`
- `results/pca/pca_summary.json` (with `formal_holdout_accessed: false`)
- `results/pca/logistic_regression_benchmark_metrics.csv`
- `results/pca/logistic_regression_benchmark_folds.csv`
- `results/pca/oof_predictions_logistic_regression.csv`
- `results/pca/oof_predictions_logistic_regression_pca.csv`

## 25. PCA Conclusion

*Instruction:* Synthesize empirical conclusions answering:
1. How many PCA input features were evaluated?
2. How many components were required for 80%, 90%, and 95% variance?
3. Which original predictors contributed most strongly to PC1–PC3?
4. Did PC1/PC2 visually separate resigned and non-resigned employees?
5. Did PCA improve, degrade, or leave Logistic Regression PR-AUC unchanged?
6. Should the six headline advanced models adopt PCA or retain native feature representations?